# Steerling-8B: Exploring the Concept Library

Steerling decomposes its representations into **known concepts** (33,732 human-interpretable features)
and **discovered concepts** (101,196 learned residual features). Each concept has an ID, a name,
and a description stored in `assets/concepts/concept_labels.parquet`.

This notebook shows how to search the concept library and inspect the top tokens related to
each concept.

## 1. Load the concept catalog

The catalog loads the bundled concept table. No model or GPU needed.

In [ ]:
from steerling import ConceptCatalog

catalog = ConceptCatalog.load()
print(len(catalog), "concepts")

## 2. Search concepts by keyword

`catalog.search()` matches against concept names and descriptions. By default it returns only
**steerable known** concepts.

In [ ]:
for c in catalog.search("Book", limit=8):
    print(f"{c.concept_id:6d}  {c.name!r}  (group: {c.group_name})")

## 3. Inspect a concept

Each concept record has the following fields:

| Field | Description |
|-------|-------------|
| `name` | Human-readable concept name |
| `description` | What the concept captures |
| `group_name` | Related concepts are grouped together (not all concepts have a group) |
| `is_steerable` | Whether the concept can be amplified or suppressed |
| `is_tone` | Concept relates to tone or style |
| `is_alignment` | Concept relates to safety or alignment |
| `is_demographic` | Concept relates to demographic attributes |

In [ ]:
CONCEPT_ID = 20353  # replace with an ID from your search

c = catalog.get(CONCEPT_ID)
print("name:       ", c.name)
print("group:      ", c.group_name)
print("steerable:  ", c.is_steerable)
print("flags:      ", f"tone={c.is_tone} alignment={c.is_alignment} demographic={c.is_demographic}")
print("description:\n", c.description)

## 4. See the top tokens related to a concept

Load the model and call `concept_top_tokens` to see the tokens most related to a concept.

Replace with `guidelabs/steerling-8b-instruct` for the instruct model.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator

model = AutoModel.from_pretrained("guidelabs/steerling-8b", trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("guidelabs/steerling-8b", trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer)

In [ ]:
for token, score in generator.concept_top_tokens(CONCEPT_ID, k=15):
    print(f"{score:6.3f}  {token!r}")